# 🧬 Biomarker Discovery Pipeline (HR-FRGS)
## Multi-Phase Hypergraph-Based Analysis

> This notebook integrates the full computational biomarker discovery pipeline.  
> **Phases 1 & 3** run in R (see instructions in their sections).  
> **Phases 2, 4, 5 & 6** run here in Python.

---

### 📋 Pipeline Architecture

| Phase | Language | Module | Description |
|-------|----------|--------|-------------|
| Phase 1 | 🔵 R | `phase1_deseq2_analysis.R` | Differential Expression Analysis (DESeq2) |
| Phase 2 | 🟢 Python | `phase2_ppi_entropy.py` | PPI Network Entropy Filtering |
| Phase 3 | 🔵 R | `Phase3.R` | WGCNA Co-expression Network |
| Phase 4 | 🟢 Python | `phase4.py` | Hypergraph Construction |
| Phase 5 | 🟢 Python | `phase5.py` | Hypergraph Diffusion & Centrality |
| Phase 6 | 🟢 Python | `phase6.py` | Fuzzy Feature Selection |

----
## 🔵 Phase 1 — Differential Expression Analysis
### *(Runs in R — execute before this notebook)*

-----
## 🟢 Phase 2 — PPI Network Entropy Filtering


In [ ]:
#!/usr/bin/env python3
"""
Phase 2: PPI Network Entropy Filtering
Detects protein complexes and filters by Graph Entropy
Author: Biomarker Discovery Pipeline
Date: 2025
"""
# from cdlib import algorithms
import pandas as pd
import numpy as np
import networkx as nx
import json
import logging
from pathlib import Path
from scipy.stats import entropy as scipy_entropy
import yaml
from collections import Counter
from collections import defaultdict
from protclus import MCODE
import tempfile
import markov_clustering as mc
import os
from sklearn.metrics.pairwise import euclidean_distances
import random
from tqdm import tqdm  # For progress bars
import random

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

class PPIEntropyFilter:
    """Filter PPI network using Graph Entropy and structural quality metrics."""
    
    def __init__(self, config_path='config/pipeline_config.yaml'):
        """Initialize with configuration."""
        self.config = self._load_config(config_path)
        self.ppi_config = self.config.get('ppi_filtering', {})
        
    def _load_config(self, config_path):
        """Load YAML configuration."""
        with open(config_path) as f:
            return yaml.safe_load(f)
    
    def run(self, degs_file, ppi_file, expr_file, output_dir='output'):
        """Execute PPI filtering pipeline."""
        logger.info("Starting PPI Entropy Filtering...")
        
        # Create output directory
        Path(output_dir).mkdir(exist_ok=True)
        
        # Load data
        logger.info("Loading input data...")
        degs = pd.read_csv(degs_file, index_col=0)
        ppi = pd.read_csv(ppi_file)
        expr = pd.read_csv(expr_file, index_col=0)
        
        deg_genes = set(degs[degs['Include_In_Network']].index)
        logger.info(f"DEGs to process: {len(deg_genes)}")
        logger.info(f"Total PPI interactions: {len(ppi)}")
        
        # Filter PPI to DEGs
        ppi_filtered = ppi[
            (ppi['Protein1'].isin(deg_genes)) & 
            (ppi['Protein2'].isin(deg_genes))
        ].copy()
        logger.info(f"PPI interactions after DEG filter: {len(ppi_filtered)}")
        
        ppi_filtered = ppi_filtered[
            ppi_filtered["Confidence_Score"] >= 0.7
        ]
        logger.info(
            f"PPI interactions after confidence score filter: "
            f"{len(ppi_filtered)}"
        )
        
        # Build network
        G = self._build_network(ppi_filtered)
        logger.info(f"Network: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
        
        clusters = self._mcode_communities_from_nx_2(G)
        logger.info(f"Clusters detected: {len(clusters)}")
        
        # Calculate entropy and filter
        quality_clusters, cluster_info = self._filter_by_entropy(G, clusters)
        logger.info(f"High-quality clusters after entropy filtering: {len(quality_clusters)}")
        
        # Extract edges from quality clusters
        filtered_ppi = self._extract_edges_from_clusters(
            G, quality_clusters, ppi_filtered
        )
        
        # Save results
        self._save_results(filtered_ppi, cluster_info, output_dir)
        
        logger.info("PPI filtering complete!")
        return filtered_ppi, cluster_info
    
    def _build_network(self, ppi_df):
        """Build NetworkX graph from PPI edges."""
        G = nx.Graph()
        
        for _, row in ppi_df.iterrows():
            G.add_edge(
                row['Protein1'], 
                row['Protein2'],
                weight=row.get('Confidence_Score', 1.0)
            )
        
        return G

    
   
    
    def _mcode_communities_from_nx_2(
        self,
        G,
        vwp=0.01,
        haircut=False,
        fluff=True,
        fluff_threshold=0.1,
        min_size=2
        ):
        """
        Cytoscape-like MCODE clustering on a NetworkX graph
        (supports overlapping complexes via fluff)

        Returns
        -------
        List[List[str]]
        """

        if G.number_of_nodes() < 2 or G.number_of_edges() == 0:
            return []

        nodes = list(G.nodes())
        n = len(nodes)

        # ---------- Stage 1: Vertex Weighting ----------
        print("Stage 1: Vertex Weighting...")
        weights = {}

        for i, v in enumerate(nodes):
            nbrs = list(G.neighbors(v))
            sub_nodes = nbrs + [v]
            subgraph = G.subgraph(sub_nodes)

            if subgraph.number_of_nodes() < 2:
                weights[v] = 0.0
                continue

            max_k = 0
            best_core = None

            for k in range(1, max(dict(subgraph.degree()).values()) + 1):
                core = nx.k_core(subgraph, k)
                if core.number_of_nodes() > 0:
                    max_k = k
                    best_core = core

            if best_core is None or best_core.number_of_nodes() < 2:
                weights[v] = 0.0
                continue

            n_nodes = best_core.number_of_nodes()
            m = best_core.number_of_edges()
            density = (2 * m) / (n_nodes * (n_nodes - 1))

            weights[v] = max_k * density

            # Show progress every ~5% or at end
            if (i + 1) % max(1, n // 20) == 0 or (i + 1) == n:
                progress = ((i + 1) / n) * 100
                print(f"Progress: {progress:.1f}% (Vertex Weighting)")

        # ---------- Stage 2: Complex Detection ----------
        print("Stage 2: Complex Detection / Expansion...")
        visited = set()
        complexes = []

        nodes_sorted = sorted(weights, key=lambda x: weights[x], reverse=True)

        def expand(seed, current):
            for nbr in G.neighbors(seed):
                if nbr in visited:
                    continue
                if weights[nbr] >= weights[seed] * (1 - vwp):
                    visited.add(nbr)
                    current.add(nbr)
                    expand(nbr, current)

        for i, v in enumerate(nodes_sorted):
            if v in visited:
                continue

            visited.add(v)
            complex_nodes = {v}
            expand(v, complex_nodes)

            if len(complex_nodes) >= min_size:
                complexes.append(complex_nodes)

            # Show progress every ~5% or at end
            if (i + 1) % max(1, n // 20) == 0 or (i + 1) == n:
                progress = ((i + 1) / n) * 100
                print(f"Progress: {progress:.1f}% (Complex Detection)")

        # ---------- Stage 3: Post-processing ----------
        print("Stage 3: Post-processing (Haircut & Fluff)...")
        final_complexes = []

        for i, c in enumerate(complexes):
            sub = G.subgraph(c)

            # Haircut (2-core)
            if haircut:
                sub = nx.k_core(sub, k=2)
                c = set(sub.nodes())

            if not c:
                continue

            # Fluff (enables overlap)
            if fluff:
                expanded = set(c)
                for u in c:
                    for nbr in G.neighbors(u):
                        if nbr in expanded:
                            continue

                        nbrhood = G.subgraph([nbr] + list(G.neighbors(nbr)))
                        n_nbr = nbrhood.number_of_nodes()
                        if n_nbr < 2:
                            continue

                        m_nbr = nbrhood.number_of_edges()
                        density = (2 * m_nbr) / (n_nbr * (n_nbr - 1))

                        if density >= fluff_threshold:
                            expanded.add(nbr)

                c = expanded

            if len(c) >= min_size:
                final_complexes.append(sorted(c))

            # Show progress
            if (i + 1) % max(1, len(complexes) // 10) == 0 or (i + 1) == len(complexes):
                progress = ((i + 1) / len(complexes)) * 100
                print(f"Progress: {progress:.1f}% (Post-processing)")

        print("MCODE completed.")
        return final_complexes
        
    
    def _calculate_graph_entropy(self, cluster_nodes, G):
        """Calculate Shannon entropy of cluster's degree distribution."""


        subgraph = G.subgraph(cluster_nodes)
        # Get the degrees from the subgraph
        degrees = [d for _, d in subgraph.degree()]

        # 1. Count frequencies of unique degrees (this is the actual distribution)
        degree_counts = Counter(degrees)
        total_nodes = len(degrees)

        # 2. Calculate probabilities p(k) = (number of nodes with degree k) / (total nodes)
        pk = np.array([count / total_nodes for count in degree_counts.values()])

        # 3. Calculate Shannon entropy
        ent = scipy_entropy(pk, base=2)
        norm_ent=ent/np.log2(len(cluster_nodes))
        return norm_ent

    
    def _calculate_cluster_quality(self, cluster_nodes, G):
        """Calculate quality metrics for cluster."""
        subgraph = G.subgraph(cluster_nodes)
        n_nodes = len(cluster_nodes)
        n_edges = subgraph.number_of_edges()
        
        # Density: actual edges / possible edges
        max_edges = n_nodes * (n_nodes - 1) / 2
        density = n_edges / max_edges if max_edges > 0 else 0
        
        # Average weight (confidence score)
        weights = [
            subgraph[u][v]['weight'] 
            for u, v in subgraph.edges()
        ]
        avg_weight = np.mean(weights) if weights else 0
        
        # Clustering coefficient (cliquishness)
        clustering = nx.average_clustering(subgraph) if n_nodes > 2 else 0
        
        # Entropy
        entropy = self._calculate_graph_entropy(cluster_nodes, G)
        # entropy = self._normalized_graph_entropy(cluster_nodes,G)
        
        # Information content score (high density + low entropy + high weight)
        ic_score = (density * 
                   avg_weight * 
                   (1 - entropy) * 
                   clustering)
        
        return {
            'density': density,
            'avg_weight': avg_weight,
            'entropy': entropy,
            'clustering': clustering,
            'ic_score': ic_score,
            'n_edges': n_edges,
            'n_nodes': n_nodes
        }
    
    def _filter_by_entropy(self, G, clusters):
        """Filter clusters by entropy and structural quality."""
        quality_clusters = []
        cluster_info = []
        
        density_threshold = self.ppi_config.get('density_threshold', 0.6)
        weight_threshold = self.ppi_config.get('average_weight_threshold', 0.7)
        entropy_threshold = self.ppi_config.get('entropy_threshold', 0.5)
        min_cluster_size = 3  # At least 3 proteins
        
        for i, cluster in enumerate(clusters):
            if len(cluster) < min_cluster_size:
                continue
            
            metrics = self._calculate_cluster_quality(cluster, G)
            
            # Apply filtering criteria
            passes_filter = (
                metrics['density'] >= density_threshold and
                metrics['avg_weight'] >= weight_threshold and
                metrics['entropy'] <= entropy_threshold 
            )
            
            info = {
                'id': i,
                'nodes': cluster,
                'n_nodes': len(cluster),
                'density': metrics['density'],
                'entropy': metrics['entropy'],
                'avg_weight': metrics['avg_weight'],
                'ic_score': metrics['ic_score'],
                'passes_filter': bool(passes_filter)
            }
            
            if passes_filter:
                quality_clusters.append(cluster)
            
            cluster_info.append(info)
            
            if (i + 1) % 50 == 0:
                logger.info(f"Processed {i + 1} clusters...")
        
        logger.info(f"Clusters passing filter: {len(quality_clusters)} / {len(clusters)}")
        
        return quality_clusters, cluster_info
    
    def _extract_edges_from_clusters(self, G, clusters, ppi_df):
        """Extract edges from quality clusters."""
        cluster_edges = set()
        
        for cluster in clusters:
            subgraph = G.subgraph(cluster)
            for u, v in subgraph.edges():
                # Normalize edge direction
                edge = tuple(sorted([u, v]))
                cluster_edges.add(edge)
        
        # Filter original PPI to keep only cluster edges
        filtered_ppi = []
        for _, row in ppi_df.iterrows():
            edge = tuple(sorted([row['Protein1'], row['Protein2']]))
            if edge in cluster_edges:
                filtered_ppi.append(row)
        
        return pd.DataFrame(filtered_ppi)
    
    def _save_results(self, filtered_ppi, cluster_info, output_dir):
        """Save results to files."""
        # Save filtered PPI
        output_file = Path(output_dir) / 'phase2_filtered_ppi.csv'
        filtered_ppi.to_csv(output_file, index=False)
        logger.info(f"Filtered PPI saved: {output_file}")
        
        # Save cluster info
        cluster_file = Path(output_dir) / 'phase2_clusters_info.json'
        with open(cluster_file, 'w') as f:
            json.dump(cluster_info, f, indent=2)
        logger.info(f"Cluster info saved: {cluster_file}")
        
        # Summary statistics
        summary = {
            'total_edges_filtered': len(filtered_ppi),
            'total_unique_nodes': len(set(
                list(filtered_ppi['Protein1']) + 
                list(filtered_ppi['Protein2'])
            )),
            'total_clusters': len([c for c in cluster_info if c['passes_filter']]),
            'average_cluster_size': np.mean([
                c['n_nodes'] for c in cluster_info if c['passes_filter']
            ])
        }
        
        logger.info(f"\nSummary Statistics:")
        for key, val in summary.items():
            logger.info(f"  {key}: {val}")


def main():
    """Main execution."""
    import argparse
    
    parser = argparse.ArgumentParser(
        description='Phase 2: PPI Network Entropy Filtering'
    )
    parser.add_argument(
        '--config',
        default='config/pipeline_config.yaml',
        help='Path to config file'
    )
    parser.add_argument(
        '--degs',
        default='output/phase1_degs.csv',
        help='DEG results file'
    )
    parser.add_argument(
        '--ppi',
        default='data/ppi_network.csv',
        help='PPI network file'
    )
    parser.add_argument(
        '--expr',
        default='data/expression_matrix.csv',
        help='Expression matrix file'
    )
    parser.add_argument(
        '--output',
        default='output',
        help='Output directory'
    )
    parser.add_argument(
        '--verbose',
        action='store_true',
        help='Verbose output'
    )
    
    args = parser.parse_args()
    
    # Run filter
    filter_obj = PPIEntropyFilter(args.config)
    filter_obj.run(args.degs, args.ppi, args.expr, args.output)


if __name__ == '__main__':
    main()


---
## 🔵 Phase 3 — WGCNA Co-expression Network Construction
### *(Runs in R — execute before Phase 4)*

---
## 🟢 Phase 4 — Hypergraph Construction

In [ ]:
#!/usr/bin/env python3
"""
Phase 4: Hypergraph Construction
Merges structural (PPI) and functional (WGCNA) layers into a dual-weighted hypergraph
Author: Biomarker Discovery Pipeline
Date: 2025
"""

import pandas as pd
import numpy as np
import json
import logging
from pathlib import Path
from scipy import sparse
import yaml
from collections import defaultdict

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)


class HypergraphConstructor:
    """Constructs dual-weighted hypergraph from PPI and WGCNA networks."""
    
    def __init__(self, config_path='config/pipeline_config.yaml'):
        """Initialize with configuration."""
        self.config = self._load_config(config_path)
        self.hg_config = self.config.get('hypergraph', {})
        
    def _load_config(self, config_path):
        """Load YAML configuration."""
        with open(config_path) as f:
            return yaml.safe_load(f)
    
    def run(self,
            struct_edges_file,
            func_adj_file,
            degs_file,
            metadata_file,
            phase3_info_file,
            output_dir='output'):
        """Execute hypergraph construction pipeline."""
        
        logger.info("Starting Hypergraph Construction...")
        Path(output_dir).mkdir(exist_ok=True)
        
        # Load data
        logger.info("Loading network data...")
        struct_edges = pd.read_csv(struct_edges_file)
        func_adj = pd.read_csv(func_adj_file, index_col=0)
        degs = pd.read_csv(degs_file, index_col=0)
        with open(metadata_file, 'r') as file:
            metadata = json.load(file)
                
        # Get DEG nodes
        deg_genes = degs[degs['Include_In_Network']].index.tolist()
        genes_dict = {gene: i for i, gene in enumerate(deg_genes)}
        logger.info(f"Total DEG nodes: {len(deg_genes)}")
        # print(genes_dict)
        # Extract structural hyperedges (PPI clusters)
        logger.info("Extracting structural hyperedges from PPI network...")
        # struct_hyperedges = self._extract_structural_hyperedges(
        #     struct_edges, genes_dict
        # )
        with open(phase3_info_file,'r') as file:
           for line in file:
               if "Soft Threshold Power:" in line:
                   phase3_info = line.split(":")[-1].strip()
                #    print(f"Extracted Power: {int(phase3_info)}")


        struct_hyperedges = self._extract_structural_hyperedges_metadata(
            metadata, genes_dict
        )
        logger.info(f"Structural hyperedges (PPI clusters): {len(struct_hyperedges)}")
        
        # Extract functional hyperedges (WGCNA modules)
        logger.info("Extracting functional hyperedges from WGCNA network...")
        func_hyperedges = self._extract_functional_hyperedges(
            func_adj, genes_dict, degs, phase3_info
        )
        logger.info(f"Functional hyperedges (co-expression modules): {len(func_hyperedges)}")
        
        # Combine hyperedges
        all_hyperedges = struct_hyperedges + func_hyperedges
        logger.info(f"Total hyperedges: {len(all_hyperedges)}")
        
        # Build incidence matrix
        logger.info("Building incidence matrix...")
        H = self._build_incidence_matrix(deg_genes, all_hyperedges)
        logger.info(f"Incidence matrix shape: {H.shape}")
        logger.info(f"Sparsity: {1 - H.nnz / (H.shape[0] * H.shape[1]):.2%}")
        
        # Create metadata
        metadata = {
            'n_nodes': len(deg_genes),
            'n_hyperedges': len(all_hyperedges),
            'hyperedges': all_hyperedges
        }
        
        # Save results
        self._save_results(
            H, genes_dict, deg_genes, metadata, all_hyperedges, output_dir
        )
        
        logger.info("Hypergraph construction complete!")
        return H, metadata
    
    def _extract_structural_hyperedges_metadata(self, metadata, genes_dict):
        """Extract protein complexes as structural hyperedges."""
        hyperedges = []
        # Find clusters that pass the filter
        min_size = self.hg_config.get('min_hyperedge_size', 3)
        
        for cluster in metadata:
            if cluster['passes_filter']:
                component = cluster['nodes']
                if len(component) >= min_size:
                    node_indices = [genes_dict[gene] for gene in component if gene in genes_dict]
                    ic_score = cluster['ic_score']
                    density = cluster['density']
                    hyperedge = {
                        'type': 'structural',
                        'nodes': sorted(node_indices),
                        # 'weight': density,
                        'weight': ic_score,
                        'size': len(component),
                        'genes': list(component)
                    }
                    hyperedges.append(hyperedge)
        logger.info(f"Found {len(hyperedges)} protein complexes")
        return hyperedges

   
    def _extract_functional_hyperedges(self, func_adj, genes_dict, degs,phase3_info):
        """Extract co-expression modules as functional hyperedges."""
        
        from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
        from scipy.spatial.distance import squareform
        
        hyperedges = []


        
        # Convert adjacency to distance matrix
        # distance = 1 - correlation
        func_adj_np = func_adj.values
        genes_list = list(func_adj.index)
        
        # Map to gene indices
        gene_indices = [genes_dict[g] for g in genes_list]
        
        
        # Use hierarchical clustering on distance matrix
        dist_matrix = 1 - np.abs(func_adj_np)  # Convert correlation to distance
        
        # Ensure distance matrix is condensed
        dist_condensed = squareform(dist_matrix)
        
        # Hierarchical clustering
        if len(genes_list) > 1:
            linkage_matrix = linkage(dist_condensed, method='average') # use average or complete method 
            
            # Cut at threshold to form modules
            modules = fcluster(linkage_matrix, t=0.7, criterion='distance') 
            
            min_size = self.hg_config.get('min_hyperedge_size', 3)
            
            # Create hyperedges from modules
            for module_id in np.unique(modules):
                module_mask = modules == module_id
                module_genes_indices = np.where(module_mask)[0]
                
                if len(module_genes_indices) >= min_size:
                    module_genes = [genes_list[i] for i in module_genes_indices]
                    node_indices = [genes_dict[g] for g in module_genes]
                    
                    # Average correlation within module
                    module_adj = func_adj_np[np.ix_(module_genes_indices, module_genes_indices)]
                    avg_weight = np.mean(module_adj[np.triu_indices_from(module_adj, k=1)])
                    
                    hyperedge = {
                        'type': 'functional',
                        'nodes': sorted(node_indices),
                        'weight': max(0, avg_weight),
                        'size': len(module_genes),
                        'genes': module_genes
                    }
                    hyperedges.append(hyperedge)
        
        logger.info(f"Found {len(hyperedges)} co-expression modules")
        return hyperedges
    
    def _build_incidence_matrix(self, genes, hyperedges):
        """Build sparse incidence matrix H (genes × hyperedges)."""
        
        n_genes = len(genes)
        n_edges = len(hyperedges)
        
        rows = []
        cols = []
        data = []
        
        for edge_idx, hyperedge in enumerate(hyperedges):
            for node_idx in hyperedge['nodes']:
                rows.append(node_idx)
                cols.append(edge_idx)
                data.append(hyperedge['weight'])
        
        H = sparse.csr_matrix(
            (data, (rows, cols)),
            shape=(n_genes, n_edges)
        )
        
        return H
    
    def _save_results(self,
                      H,
                      genes_dict,
                      genes_list,
                      metadata,
                      hyperedges,
                      output_dir):
        """Save hypergraph construction results."""
        
        output_path = Path(output_dir)
        
        # Save incidence matrix
        incidence_file = output_path / 'phase4_hypergraph_incidence.npz'
        sparse.save_npz(incidence_file, H)
        logger.info(f"Incidence matrix saved: {incidence_file}")
        
        # Save metadata
        metadata_file = output_path / 'phase4_hypergraph_metadata.json'
        with open(metadata_file, 'w') as f:
            json.dump(metadata, f, indent=2, default=str)
        logger.info(f"Metadata saved: {metadata_file}")
        
        # Save node-hyperedge mapping
        node_map_data = []
        for edge_idx, hyperedge in enumerate(hyperedges):
            for node_idx in hyperedge['nodes']:
                node_map_data.append({
                    'Gene_Index': node_idx,
                    'Gene': genes_list[node_idx],
                    'Hyperedge_ID': edge_idx,
                    'Hyperedge_Type': hyperedge['type'],
                    'Hyperedge_Weight': hyperedge['weight']
                })
        
        node_map_df = pd.DataFrame(node_map_data)
        node_map_file = output_path / 'phase4_node_hyperedge_map.csv'
        node_map_df.to_csv(node_map_file, index=False)
        logger.info(f"Node-hyperedge map saved: {node_map_file}")
        
        # Summary statistics
        logger.info("\nHypergraph Summary:")
        logger.info(f"  Nodes (genes): {len(genes_list)}")
        logger.info(f"  Hyperedges: {len(hyperedges)}")
        logger.info(f"  Structural hyperedges: {sum(1 for e in hyperedges if e['type']=='structural')}")
        logger.info(f"  Functional hyperedges: {sum(1 for e in hyperedges if e['type']=='functional')}")
        logger.info(f"  Average hyperedge size: {np.mean([e['size'] for e in hyperedges]):.1f}")
        logger.info(f"  Max hyperedge size: {max([e['size'] for e in hyperedges])}")
        logger.info(f"  Min hyperedge size: {min([e['size'] for e in hyperedges])}")


def main():
    """Main execution."""
    import argparse
    
    parser = argparse.ArgumentParser(
        description='Phase 4: Hypergraph Construction'
    )
    parser.add_argument(
        '--config',
        default='config/pipeline_config.yaml',
        help='Path to config file'
    )
    parser.add_argument(
        '--struct-edges',
        default='output/phase2_filtered_ppi.csv',
        help='Structural edges (PPI)'
    )
    parser.add_argument(
        '--func-adj',
        default='output/phase3_pruned_adjacency.csv',
        help='Functional adjacency (WGCNA)'
    )
    parser.add_argument(
        '--degs',
        default='output/phase1_degs.csv',
        help='DEG results'
    )
    parser.add_argument(
        '--metadata',
        default='output/phase2_clusters_info.json',
        help= 'Protein cluster metadata'
    )
    parser.add_argument(
        '--phase3_power_info',
        default='output/phase3_power_info.txt',
        help='phase3_info'
    )
    parser.add_argument(
        '--output',
        default='output',
        help='Output directory'
    )
    
    args = parser.parse_args()
    
    # Run construction
    constructor = HypergraphConstructor(args.config)
    constructor.run(
        args.struct_edges,
        args.func_adj,
        args.degs,
        args.metadata,
        args.phase3_power_info,
        args.output
    )


if __name__ == '__main__':
    main()


---
## 🟢 Phase 5 — Hypergraph Diffusion & Centrality Analysis

In [ ]:
#!/usr/bin/env python3
"""
Phase 5: Hypergraph Diffusion & Centrality Analysis
Implements Random Walk with Restart and Hypergraph Betweenness Centrality
Author: Biomarker Discovery Pipeline
Date: 2025
"""

import pandas as pd
import numpy as np
import networkx as nx
from scipy import sparse
import json
import logging
from pathlib import Path
import yaml
from collections import defaultdict
import warnings

warnings.filterwarnings('ignore')

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)


class HypergraphDiffusionAnalyzer:
    """Performs diffusion-based biomarker discovery on hypergraphs."""
    
    def __init__(self, config_path='config/pipeline_config.yaml'):
        """Initialize with configuration."""
        self.config = self._load_config(config_path)
        self.diff_config = self.config.get('diffusion', {})
        self.scoring_config = self.config.get('scoring', {})
        
    def _load_config(self, config_path):
        """Load YAML configuration."""
        with open(config_path) as f:
            return yaml.safe_load(f)
    
    def run(self, 
            hypergraph_file,
            metadata_file,
            node_map_file,
            degs_file,
            probmap_file,
            output_dir='output'):
        """Execute hypergraph diffusion pipeline."""
        
        logger.info("Starting Hypergraph Diffusion Analysis...")
        Path(output_dir).mkdir(exist_ok=True)
        
        # Load data
        logger.info("Loading hypergraph data...")
        H = sparse.load_npz(hypergraph_file)
        
        with open(metadata_file,'r') as f:
            meta = json.load(f)
        
        node_map = pd.read_csv(node_map_file, index_col=0)
        genes = node_map.index.astype(int).tolist()
        degs = pd.read_csv(degs_file, index_col=0)
        filtered_degs=degs[degs['Include_In_Network']].index.tolist()
        gene_dict={gene: i for i, gene in enumerate(filtered_degs)}
        reverse_map = {index: name for name, index in gene_dict.items()}
        # This creates the 215 x 33 matrix you need
        logger.info("Slicing H to match specific gene indices...")
        H_filtered=H[genes, :]


        probmap=pd.read_table(probmap_file)
        id_to_symbol = dict(zip(probmap['id'], probmap['gene']))

        # Debug prints for verification
        logger.info(f"Hypergraph: {H.shape[0]} nodes, {H.shape[1]} hyperedges")
        logger.info(f"Number of genes from node_map: {len(node_map.index)}")
        logger.info(f"Number of genes from degs (full): {len(degs.index)}")
        logger.info(f"Number of genes used for H: {len(genes)}")
        
        # Build transition matrix
        logger.info("Building transition matrix...")
        P = self._build_transition_matrix(H_filtered)
        
        # Random Walk with Restart
        logger.info("Running Random Walk with Restart...")
        diffusion_scores = self._random_walk_restart(P, len(genes))
        
        # Hypergraph Betweenness Centrality
        logger.info("Computing Hypergraph Betweenness Centrality...")
        hbc_scores = self._compute_hypergraph_betweenness(H_filtered, genes)
        
        # Combine scores
        logger.info("Computing composite biomarker scores...")
        biomarker_scores = self._compute_biomarker_scores(
            diffusion_scores, 
            hbc_scores,
            genes,
            reverse_map,
            degs,
            id_to_symbol,
            meta
        )
        
        # Save results
        self._save_results(
            biomarker_scores,
            diffusion_scores,
            hbc_scores,
            genes,
            output_dir
        )
        
        logger.info("Diffusion analysis complete!")
        return biomarker_scores
    
    def _build_transition_matrix(self, H):
        """
        Build hypergraph transition matrix.
        P = Dv^-1 * H * W * De^-1 * H^T
        
        Where:
        - Dv: diagonal matrix of node weighted degrees
        - H: incidence matrix
        - W: diagonal matrix of hyperedge weights
        - De: diagonal matrix of hyperedge sizes
        """
        H_dense = H.toarray() if sparse.issparse(H) else H
        
        # Initialize weights (assuming W is identity for now)
        w = np.ones(H.shape[1])
        W = sparse.diags(w)
        
        # Node weighted degrees
        Dv_inv = np.array(H @ w)
        Dv_inv = 1.0 / (Dv_inv + 1e-10)
        Dv_inv_mat = sparse.diags(Dv_inv)
        
        # Hyperedge sizes
        De_inv = np.array(H.sum(axis=0)).flatten()
        De_inv = 1.0 / (De_inv + 1e-10)
        De_inv_mat = sparse.diags(De_inv)
        
        # Compute transition matrix
        H_sparse = sparse.csr_matrix(H_dense)
        P = Dv_inv_mat @ H_sparse @ W @ De_inv_mat @ H_sparse.T
        # print(P)
        return P
    
    def _random_walk_restart(self, P, n_nodes):
        """
        Implement Random Walk with Restart.
        r_t+1 = (1-c) * P * r_t + c * r_0
        """
        c = float(self.diff_config.get('restart_probability', 0.1))
        max_iter = int(self.diff_config.get('max_iterations', 1000))
        threshold = float(self.diff_config.get('convergence_threshold', 1e-6))
        
        # Initialize with uniform distribution
        r = np.ones(n_nodes) / n_nodes
        # print(r)
        r_0 = r.copy()
        
        logger.info(f"RWR parameters: c={c}, max_iter={max_iter}, threshold={threshold}")
        
        for iteration in range(max_iter):
            r_new = (1 - c) * (P @ r) + c * r_0
            
            # Check convergence
            diff = np.linalg.norm(r_new - r)
            if diff < threshold:
                logger.info(f"RWR converged at iteration {iteration}, diff={diff:.2e}")
                return r_new
            
            r = r_new
            
            if (iteration + 1) % 100 == 0:
                logger.info(f"  Iteration {iteration + 1}, diff={diff:.2e}")
        
        logger.warning(f"RWR did not converge after {max_iter} iterations")
        return r
    
    def _compute_hypergraph_betweenness(self, H, genes):
        """
        Compute Hypergraph Betweenness Centrality.
        Approximated using random sampling for efficiency.
        """
        n_nodes = len(genes)
        hbc = np.zeros(n_nodes)
        
        # Build undirected graph from hypergraph
        # For each hyperedge, create a clique among its members
        H_dense = H.toarray() if sparse.issparse(H) else H
        G = nx.Graph()
        G.add_nodes_from(range(n_nodes))
        
        for e in range(H_dense.shape[1]):
            members = np.where(H_dense[:, e] > 0)[0]
            if len(members) > 1:
                # Add weighted edges (weight = hyperedge weight)
                for i in range(len(members)):
                    for j in range(i + 1, len(members)):
                        u, v = members[i], members[j]
                        if G.has_edge(u, v):
                            G[u][v]['weight'] += 1.0
                        else:
                            G.add_edge(u, v, weight=1.0)
        
        # Compute betweenness centrality
        # Use approximate algorithm for large graphs
        if G.number_of_edges() > 0:
            try:
                hbc_dict = nx.betweenness_centrality(
                    G, 
                    normalized=True,
                    weight='weight'
                )
                hbc = np.array([hbc_dict.get(i, 0) for i in range(n_nodes)])
            except:
                logger.warning("Failed to compute betweenness centrality, using degree")
                degree = np.array([G.degree(i, weight='weight') for i in range(n_nodes)])
                hbc = degree / (np.max(degree) + 1e-10) if np.max(degree) > 0 else degree
        
        return hbc
    
    def _compute_biomarker_scores(self, 
                              diffusion_scores,
                              hbc_scores,
                              genes,
                              gene_dict,
                              degs,
                              id_to_symbol,
                              meta):
        """Compute composite biomarker potential scores."""
        
        alpha = float(self.scoring_config.get('alpha', 0.5))
        
        # Normalize scores to 0-1 range
        diff_norm = (diffusion_scores - diffusion_scores.min()) / (
            diffusion_scores.max() - diffusion_scores.min() + 1e-10
        )
        hbc_norm = (hbc_scores - hbc_scores.min()) / (
            hbc_scores.max() - hbc_scores.min() + 1e-10
        )
        
        # Compute ranks
        diff_rank = pd.Series(diff_norm).rank(pct=True).values
        hbc_rank = pd.Series(hbc_norm).rank(pct=True).values
        
        # Composite score
        bps = alpha * diff_rank + (1 - alpha) * hbc_rank
        
        # Pre-compute gene → list of (hyperedge_index, type) for fast lookup
        gene_to_modules = {}
        for idx, hyperedge in enumerate(meta['hyperedges']):
            h_type = hyperedge['type']  # 'structural' or 'functional'
            for gene_id in hyperedge['genes']:
                if gene_id not in gene_to_modules:
                    gene_to_modules[gene_id] = []
                gene_to_modules[gene_id].append((idx, h_type))
        # print(gene_to_modules)
        # Determine membership category and list of module IDs
        gene_membership = {}
        gene_structural_ids = {}
        gene_functional_ids = {}
        for gene_id, modules in gene_to_modules.items():
            types = {t for _, t in modules}
            structural_module_ids = [str(i) for i, t in modules if t == 'structural']
            functional_module_ids = [str(i) for i, t in modules if t == 'functional']
            
            if 'structural' in types and 'functional' in types:
                category = 'both'
            elif 'structural' in types:
                category = 'structural_only'
            else:
                category = 'functional_only'
            
            gene_membership[gene_id] = category
            gene_structural_ids[gene_id] = ', '.join(sorted(structural_module_ids))  # e.g., "0, 23, 45"
            gene_functional_ids[gene_id] = ', '.join(sorted(functional_module_ids))  # e.g., "0, 23, 45"
        
            # Create results dataframe
        results = []
        for i, gene in enumerate(genes):
                ensembl_id = gene_dict[gene]
                gene_symbol = id_to_symbol.get(ensembl_id, ensembl_id)
                
                # Get DEG info
                deg_info = degs.loc[ensembl_id] if ensembl_id in degs.index else pd.Series()
                
                # Module membership
                membership_category = gene_membership.get(ensembl_id, 'none')
                structural_ids = gene_structural_ids.get(ensembl_id, '')
                functional_ids = gene_functional_ids.get(ensembl_id, '')
                
                result = {
                    'Gene': ensembl_id,
                    'Gene_Symbol': gene_symbol,
                    'Diffusion_Score': float(diff_norm[i]),
                    'Rank_Diffusion': int(np.argsort(-diff_norm)[np.where(np.argsort(-diff_norm) == i)[0][0]] + 1),
                    'HBC_Score': float(hbc_norm[i]),
                    'Rank_HBC': int(np.argsort(-hbc_norm)[np.where(np.argsort(-hbc_norm) == i)[0][0]] + 1),
                    'BiomarkerPotentialScore': float(bps[i]),
                    'Expression_LogFC': float(deg_info.get('log2FoldChange', 0)),
                    'P_Value_Adj': float(deg_info.get('padj', 1.0)),
                    'Module_Type': membership_category,          # new column 1: "structural_only" | "functional_only" | "both" | "none"
                    'structural_module_ids': structural_ids,     # new column 2: comma-separated list of structural hyperedge indices (e.g., "0, 23")
                    'functional_module_ids': functional_ids      # new column 3: comma-separated list of functional hyperedge indices (e.g., "0, 23")
                }
                results.append(result)
            
        # Convert to dataframe
        results_df = pd.DataFrame(results)
        
        # Remove duplicates, keeping the first occurrence
        results_df = results_df.drop_duplicates(subset=['Gene'], keep='first')
        
        # Sort and rank
        results_df = results_df.sort_values('BiomarkerPotentialScore', ascending=False)
        results_df['Final_Rank'] = range(1, len(results_df) + 1)
        
        return results_df
    
    def _save_results(self, 
                     biomarker_scores,
                     diffusion_scores,
                     hbc_scores,
                     genes,
                     output_dir):
        """Save all results to files."""
        
        output_path = Path(output_dir)
        
        # Save main biomarker file
        biomarker_file = output_path / 'BIOMARKERS_FINAL.csv'
        biomarker_scores.to_csv(biomarker_file, index=False)
        logger.info(f"Final biomarkers saved: {biomarker_file}")
        
        # Save diffusion scores
        diffusion_file = output_path / 'phase5_diffusion_scores.csv'
        diff_df = pd.DataFrame({
            'Gene': genes,
            'Diffusion_Score': diffusion_scores,
            'Rank': np.argsort(-diffusion_scores) + 1
        })
        diff_df.to_csv(diffusion_file, index=False)
        logger.info(f"Diffusion scores saved: {diffusion_file}")
        
        # Save centrality scores
        centrality_file = output_path / 'phase5_centrality_scores.csv'
        hbc_df = pd.DataFrame({
            'Gene': genes,
            'HBC_Score': hbc_scores,
            'Rank': np.argsort(-hbc_scores) + 1
        })
        hbc_df.to_csv(centrality_file, index=False)
        logger.info(f"Centrality scores saved: {centrality_file}")
        
        # Print top biomarkers
        logger.info("\nTop 20 Biomarkers:")
        logger.info("=" * 80)
        for idx, row in biomarker_scores.head(20).iterrows():
            logger.info(
                f"{row['Final_Rank']:3d}. {row['Gene_Symbol']:15s} "
                f"BPS={row['BiomarkerPotentialScore']:.4f} "
                f"DiffScore={row['Diffusion_Score']:.4f} "
                f"HBC={row['HBC_Score']:.4f} "
                f"LogFC={row['Expression_LogFC']:+.2f}"
            )


def main():
    """Main execution."""
    import argparse
    
    parser = argparse.ArgumentParser(
        description='Phase 5: Hypergraph Diffusion & Centrality Analysis'
    )
    parser.add_argument(
        '--config',
        default='config/pipeline_config.yaml',
        help='Path to config file'
    )
    parser.add_argument(
        '--hypergraph',
        default='output/phase4_hypergraph_incidence.npz',
        help='Hypergraph incidence matrix'
    )
    parser.add_argument(
        '--metadata',
        default='output/phase4_hypergraph_metadata.json',
        help='Hypergraph metadata file'
    )
    parser.add_argument(
        '--node-map',
        default='output/phase4_node_hyperedge_map.csv',
        help='Node-to-hyperedge mapping'
    )
    parser.add_argument(
        '--degs',
        default='output/phase1_degs.csv',
        help='DEG file'
    )
    parser.add_argument(
        '--probmap',
        default='data/gencode.v36.annotation.gtf.gene.probemap',
        help='PROBMAP file'
    )
    parser.add_argument(
        '--output',
        default='output',
        help='Output directory'
    )
    parser.add_argument(
        '--verbose',
        action='store_true',
        help='Verbose output'
    )
    
    args = parser.parse_args()
    
    # Run analysis
    analyzer = HypergraphDiffusionAnalyzer(args.config)
    analyzer.run(
        args.hypergraph,
        args.metadata,
        args.node_map,
        args.degs,
        args.probmap,
        args.output
    )


if __name__ == '__main__':
    main()

---
## 🟢 Phase 6 — Fuzzy Feature Selection

In [ ]:
import logging
import numpy as np
import pandas as pd
import skfuzzy as fuzz
import matplotlib.pyplot as plt
from typing import List, Tuple, Dict
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import KFold

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class HypergraphFeatureSelector:
    def __init__(self, expr_matrix: np.ndarray, bps: np.ndarray, labels: np.ndarray):
        """Initializes the selector, caching shared matrices to avoid redundant memory usage."""
        self.expr = expr_matrix.astype(np.float32)
        self.labels = labels
        self.classes = np.unique(labels)
        
        # Pre-compute one-hot class membership matrix (n_samples, n_classes)
        self.mu_D = (self.labels[:, None] == self.classes).astype(float)
        
        # Normalize weights safely
        bps_clean = np.clip(bps, a_min=0.0, a_max=None) + 1e-10
        self.W = bps_clean / bps_clean.sum()
        logger.info(f"Initialized Selector. Sum(W)={self.W.sum():.4f}, Max(W)={self.W.max():.4f}")

    def fcm_grouping(self, n_clusters: int, m: float = 2.0, threshold: float=0.20) -> Tuple[List[Dict], np.ndarray]:
        """
        Clusters features using Fuzzy C-Means and extracts representatives.
        Allows for overlapping clusters based on a membership threshold.
        """
        # 1. Prepare Features
        feats = StandardScaler().fit_transform(self.expr.T)
        bps_norm = MinMaxScaler().fit_transform(self.W.reshape(-1, 1))
        feats = np.hstack([feats, bps_norm])
        
        # 2. Run Fuzzy C-Means
        # u is the membership matrix of shape (n_clusters, n_genes)
        _, u, _, _, _, _, _ = fuzz.cluster.cmeans(
            feats.T, c=n_clusters, m=m, error=0.005, maxiter=1000, init=None, seed=42
        )
        
        groups, reps = [], []
        
        # 3. Process each cluster to allow overlap
        for cl in range(n_clusters):
            # CHANGE: Instead of hard_labels, we find all genes where 
            # membership in THIS cluster 'cl' is >= threshold
            cluster_membership = u[cl, :]
            idx = np.where(cluster_membership >= threshold)[0]
            
            if not len(idx): 
                continue
                
            # The 'Representative' is the gene with the absolute highest 
            # membership specifically for this cluster
            rep = idx[np.argmax(cluster_membership[idx])]
            
            groups.append({
                'cluster_id': cl, 
                'genes': idx.tolist(), 
                'rep': int(rep), 
                'size': len(idx)
            })
            reps.append(rep)
            
        # We use np.unique because in overlapping clustering, 
        # the same gene could theoretically be a representative for two clusters
        unique_reps = np.unique(reps)
    
        logger.info(f"Grouped into {len(groups)} overlapping clusters.")
        logger.info(f"Total gene assignments: {sum(g['size'] for g in groups)} (Total Genes: {feats.shape[0]})")
        
        return groups, unique_reps
    
    def compute_ipd(self, B: List[int]) -> float:
        """Fully vectorized computation of Hypergraph Inner Product Dependency (IPD)."""
        if not B: return 0.0
        
        expr_B = self.expr[:, B]
        diffs = np.abs(expr_B[:, None, :] - expr_B[None, :, :])
        dist = np.sum(diffs * self.W[B], axis=-1)
        sigma = np.std(dist[~np.eye(len(dist), dtype=bool)]) or 1e-6
        
        R = np.exp(-dist / sigma)
        np.fill_diagonal(R, 0.0) 
        lower = np.min(np.maximum(1 - R[:, :, None], self.mu_D[None, :, :]), axis=1)
        
        gamma_c = np.mean(np.max(lower, axis=1))
        overlap = 0.5 * np.mean(np.sum(lower, axis=1)**2 - np.sum(lower**2, axis=1))
        
        return float(np.clip(gamma_c - overlap, 0.0, 1.0))

    def _prune(self, S: List[int]) -> List[int]:
        """Strict Pruning: Remove gene ONLY if removal improves the IPD score."""
        pruned_S = list(S)
        
        # We iterate backwards to check if the newest additions can be optimized
        for g in reversed(S):
            if len(pruned_S) <= 1: break
            
            current_gamma = self.compute_ipd(pruned_S)
            temp_S = [x for x in pruned_S if x != g]
            gamma_without_g = self.compute_ipd(temp_S)
            
            # THE LOGIC: If removing g makes the score BETTER, then g is noise.
            # If removing g makes it worse OR stays the same, we KEEP g.
            if gamma_without_g > current_gamma:
                pruned_S = temp_S
                logger.info(f"Pruned gene {g}: IPD improved {current_gamma:.4f} -> {gamma_without_g:.4f}")
                
        return pruned_S

    def _check_halting(self, hist: List[Tuple[int, float]], stab_hist: List[float]) -> Tuple[bool, int, str]:
        """Halts primarily on stability drops, as IPD plateauing is now allowed."""
        if len(hist) < 5: return False, -1, ""

        # Stability is the best signal for your 'inclusive' logic.
        # If stability drops below 0.7 or drops significantly (>15%) compared to recent history.
        if len(stab_hist) >= 3:
            current_stab = stab_hist[-1]
            prev_avg_stab = np.mean(stab_hist[:-1])
            if current_stab < 0.7 or current_stab < (0.85 * prev_avg_stab):
                return True, len(hist), "Stability collapse"

        # Curvature still useful to find the 'Mathematical Knee' for the plot marker
        sizes, gammas = np.array([h[0] for h in hist]), np.array([h[1] for h in hist])
        xn, yn = (sizes - sizes.min()) / (np.ptp(sizes) + 1e-12), (gammas - gammas.min()) / (np.ptp(gammas) + 1e-12)
        dy = (yn[2:] - yn[:-2]) / (xn[2:] - xn[:-2] + 1e-12)
        d2y = (yn[2:] - 2 * yn[1:-1] + yn[:-2]) / ((xn[1:-1] - xn[:-2])**2 + 1e-12)
        kappa = np.abs(d2y) / ((1 + dy**2)**1.5 + 1e-12)
        knee_idx = int(np.argmax(kappa) + 1) if len(kappa) > 0 else len(hist)

        # We return the full history length as K_opt because you want to 'count in' 
        # all genes that maintain the score, but we label the reason if we stop.
        return False, knee_idx, ""

    def plot_diagnostics(self, hist: List[Tuple[int, float]], stab_hist: List[float], stab_sizes: List[int], save_path: str):
        """Enhanced 1x3 diagnostic plot for 'Maintain-or-Increase' logic."""
        if len(hist) < 2:
            logger.warning("Not enough history for plots.")
            return

        sizes, gammas = np.array([h[0] for h in hist]), np.array([h[1] for h in hist])
        
        # Normalize for curvature calculation (NumPy 2.0 compatible)
        xn = (sizes - sizes.min()) / (np.ptp(sizes) + 1e-12)
        yn = (gammas - gammas.min()) / (np.ptp(gammas) + 1e-12)
        
        # Calculate Curvature (Kappa)
        dy = np.gradient(yn, xn)
        d2y = np.gradient(dy, xn)
        kappa = np.abs(d2y) / ((1 + dy**2)**1.5 + 1e-12)
        knee_idx = np.argmax(kappa)

        fig, axes = plt.subplots(1, 2, figsize=(20, 6))

        # 1. Dependency Curve: Shows the plateau where genes 'maintain' the score
        axes[0].plot(sizes, gammas, 'b-', alpha=0.6, label='IPD Path')
        axes[0].scatter(sizes, gammas, c='blue', s=30, label='Included Genes')
        # Highlight the Knee (The point of diminishing returns)
        axes[0].axvline(sizes[knee_idx], color='red', linestyle='--', alpha=0.5, label='Math Knee')
        axes[0].set(xlabel='Subset Size |S|', ylabel='IPD (γ)', title='Dependency Growth & Plateau')
        axes[0].legend()
        axes[0].grid(True, alpha=0.2)

        # 2. Curvature Profile: Shows where the 'bend' happens
        axes[1].plot(sizes, kappa / (kappa.max() + 1e-12), color='orange', lw=2, label='Curvature (κ)')
        axes[1].fill_between(sizes, kappa / (kappa.max() + 1e-12), color='orange', alpha=0.1)
        axes[1].axvline(sizes[knee_idx], color='red', linestyle='--')
        axes[1].set(xlabel='|S|', ylabel='Normalized Curvature', title='Information Gain Knee')
        axes[1].legend()
        axes[1].grid(True, alpha=0.2)

        plt.suptitle(f"Hypergraph Selector Diagnostics (Final |S| = {len(sizes)})", fontsize=14)
        plt.tight_layout()
        plt.savefig(save_path, dpi=300)
        plt.close()
        logger.info(f"Updated diagnostic plots saved to {save_path}")

    def greedy_search(self, candidates: np.ndarray, max_iter=200, check_stability=True) -> Tuple[List[int], List, List, List, int]:
        """Forward greedy search to find the optimal feature subset."""
        S, hist, stab_hist, stab_sizes = [], [], [], []
        for i in range(max_iter):
            best_a, best_delta = None, -np.inf
            base_ipd = self.compute_ipd(S) if S else 0.0
            
            for a in set(candidates) - set(S):
                delta = self.compute_ipd(S + [a]) - base_ipd
                if delta >= best_delta:
                    best_delta, best_a = delta, a
                    
            if best_a is None or best_delta < 0: break
            S = self._prune(S + [best_a])
            hist.append((len(S), self.compute_ipd(S)))
            
            if check_stability and i % 4 == 0 and len(S) >= 5:
                stab = self._jaccard_stability(S)
                stab_hist.append(stab)
                stab_sizes.append(len(S))
                logger.info(f"Iter {i} | |S|={len(S)} | IPD={hist[-1][1]:.4f} | Stab={stab:.4f}")
            
            halt, k_opt, reason = self._check_halting(hist, stab_hist)
            if halt:
                logger.info(f"HALTED at K={k_opt} ({reason})")
                return S[:k_opt], hist, stab_hist, stab_sizes, k_opt
                
        return S, hist, stab_hist, stab_sizes, len(S)


def main():
    # 1. Configuration & Data Load
    EXPR_PATH = r'data/expression_matrix.csv'
    BIO_PATH  = r"output/BIOMARKERS_FINAL.csv"
    META_PATH = r"data/metadata.csv"
    
    expr_df = pd.read_csv(EXPR_PATH, index_col=0)
    expr_df=np.log2(expr_df+1)

    bio_df = pd.read_csv(BIO_PATH)
    meta_df = pd.read_csv(META_PATH)
    
    labels = (meta_df['Status'] == "Tumor").astype(int).values
    genes, bps = bio_df['Gene'].values, bio_df['BiomarkerPotentialScore'].values
    
    # 2. Align Data & Init Pipeline
    expr_df = expr_df.reindex(index=genes).dropna()
    selector = HypergraphFeatureSelector(expr_df.values.T, bps, labels)
    
    # 3. FCM Grouping & Best Candidate Extraction
    groups, reps = selector.fcm_grouping(n_clusters=max(1, len(genes)//50))
    best_cands = [max(g['genes'], key=lambda x: selector.compute_ipd([x])) for g in groups]
    
    # 4. Autonomous Greedy Search & Plotting
    logger.info("Starting Autonomous Greedy Search...")
    S_final, hist_final, stab_hist_final, stab_sizes_final, k_opt = selector.greedy_search(np.array(best_cands))
    logger.info(f"Final Selection: {S_final} (IPD: {selector.compute_ipd(S_final):.4f})")
    
    # Generate Phase 1 Plots
    selector.plot_diagnostics(hist_final, stab_hist_final, stab_sizes_final, "output/phase1_diagnostics.png")
    combined_pool = np.unique([g for grp in groups if any(s in grp['genes'] for s in S_final) for g in grp['genes']])
    
    # 6. Extract Biomarkers
    cluster_map = {g: grp['cluster_id'] for grp in groups for g in grp['genes']}
    extracted = bio_df.iloc[combined_pool].copy()
    extracted['cluster_id'] = extracted.index.map(cluster_map)
    extracted['is_selected'] = extracted.index.isin(S_final)
    
    extracted.to_csv('output/extracted_biomarkers.csv', index=False)
    logger.info("Pipeline Complete. Data saved to output/extracted_biomarkers.csv")

if __name__ == "__main__":
    main()